# Notebook 02 — Bayesian Thinking

**The core idea:** You start with a belief. You see new data. You update your belief. Repeat.

This is not just a statistics concept — it's how good traders think. Every time the market gives you new information (a big move, an earnings surprise, a vol spike), you should be updating your estimate of what's going on.

By the end of this notebook you will:
- Understand Bayes' theorem and be able to compute it from scratch
- Run a prior → posterior update loop
- Apply Bayesian reasoning to updating a vol estimate

---

**Loop reminder:** CONCEPT → CODE → CHALLENGE → ASSERT → REFLECT

In [ ]:
# Setup — run this first
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
print('Ready.')

---
## Part 1 — The Problem with Static Thinking

Suppose you wake up and someone tells you: *"IBM moved 4% today."*

How unusual is that? Well, it depends on what you already knew:
- Was IBM reporting earnings today? Then a 4% move is unremarkable.
- Was it a quiet Tuesday with no news? Then a 4% move is a significant signal — maybe something broke.

The move alone doesn't tell you what's happening. You have to combine the move with what you already believed to figure out what it means.

**That updating process is exactly what Bayes' theorem does.**

---

## Part 2 — Bayes' Theorem

The formula:

$$P(H | D) = \frac{P(D | H) \cdot P(H)}{P(D)}$$

Translated into plain English:

| Symbol | Meaning | Plain English |
|--------|---------|---------------|
| `P(H)` | Prior | Your belief *before* seeing the data |
| `P(D\|H)` | Likelihood | How probable is this data *if H is true?* |
| `P(D)` | Marginal | How probable is this data overall? (normalizer) |
| `P(H\|D)` | Posterior | Your updated belief *after* seeing the data |

The output `P(H|D)` is called the **posterior**. It becomes your new prior the next time you get data.

---

### Concrete Example: Medical Test

A disease affects 1% of the population.  
A test for it is 95% accurate (true positive rate).  
But the test has a 5% false positive rate (says positive when you don't have it).

**You test positive. What's the actual probability you have the disease?**

Most people say ~95%. The real answer is much lower. Let's compute it.

In [ ]:
# === CONCEPT: Medical Test Example ===

# Known values
p_disease       = 0.01   # 1% of population has it (prior)
p_positive_given_disease   = 0.95   # true positive rate (sensitivity)
p_positive_given_no_disease = 0.05  # false positive rate

# Complement
p_no_disease = 1 - p_disease

# P(D) — total probability of testing positive
# This accounts for all the ways you could get a positive test:
#   (have disease AND test positive) OR (no disease AND test positive anyway)
p_positive = (p_positive_given_disease * p_disease) + \
             (p_positive_given_no_disease * p_no_disease)

# Bayes' theorem — posterior
p_disease_given_positive = (p_positive_given_disease * p_disease) / p_positive

print(f'Prior probability of disease:     {p_disease:.1%}')
print(f'P(positive test) overall:         {p_positive:.1%}')
print(f'P(disease | positive test):       {p_disease_given_positive:.1%}')
print()
print('Even with a 95% accurate test, a positive result only means')
print(f'~{p_disease_given_positive:.0%} chance of actually having the disease.')
print('Why? Because the disease is rare. Most positives are false positives.')

In [ ]:
# === CONCEPT: Visualize what's happening ===

# Imagine testing 10,000 people
n = 10_000

have_disease    = int(n * p_disease)          # 100 people
no_disease      = n - have_disease             # 9,900 people

true_positives  = int(have_disease * p_positive_given_disease)     # ~95
false_positives = int(no_disease   * p_positive_given_no_disease)  # ~495

total_positives = true_positives + false_positives

fig, ax = plt.subplots(figsize=(9, 5))

categories = ['True Positives\n(have disease,\ntest positive)',
              'False Positives\n(no disease,\ntest positive)']
counts = [true_positives, false_positives]
colors = ['#2ecc71', '#e74c3c']

bars = ax.bar(categories, counts, color=colors, edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', fontsize=13, fontweight='bold')

ax.set_title(f'Out of {n:,} people tested — {total_positives} test positive', fontsize=14)
ax.set_ylabel('Number of people')
ax.set_ylim(0, 600)
ax.annotate(f'Only {true_positives}/{total_positives} = {true_positives/total_positives:.1%} of positives actually have the disease',
            xy=(0.5, 0.85), xycoords='axes fraction', ha='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.show()

---
### Challenge 1 — Change the Prevalence

The disease is rare (1%) which is why the positive test is so misleading. Now imagine a different scenario:

- **Scenario A:** The disease affects 30% of the population (common illness)
- **Scenario B:** The disease affects 0.1% (extremely rare)

**Your task:** Copy the calculation above and run it with both new prevalence rates. Then answer:

1. For scenario A, what is `P(disease | positive test)`?
2. For scenario B, what is `P(disease | positive test)`?
3. Write a comment explaining why the prior (prevalence) has such a big impact on the result.

**Trading connection to keep in mind:** In trading, the "prior" is your base rate. How often does a stock with this profile actually have a true catalyst vs. random noise? A signal means much more when your prior says the signal is credible.

In [ ]:
# === CHALLENGE 1 — Modify this cell ===

# Scenario A — common disease
p_disease_A = 0.01   # <-- CHANGE THIS to 0.30
p_positive_given_disease_A   = 0.95  # keep the same
p_positive_given_no_disease_A = 0.05  # keep the same

p_positive_A = (p_positive_given_disease_A * p_disease_A) + \
               (p_positive_given_no_disease_A * (1 - p_disease_A))
posterior_A = (p_positive_given_disease_A * p_disease_A) / p_positive_A

# Scenario B — very rare disease
p_disease_B = 0.01   # <-- CHANGE THIS to 0.001
p_positive_given_disease_B   = 0.95
p_positive_given_no_disease_B = 0.05

p_positive_B = (p_positive_given_disease_B * p_disease_B) + \
               (p_positive_given_no_disease_B * (1 - p_disease_B))
posterior_B = (p_positive_given_disease_B * p_disease_B) / p_positive_B

print(f'Scenario A posterior (30% prevalence):  {posterior_A:.1%}')
print(f'Scenario B posterior (0.1% prevalence): {posterior_B:.1%}')

# YOUR EXPLANATION:
# Why does prior prevalence have such a big effect on the posterior?
# [Write your answer here as a comment]

In [ ]:
# === ASSERT 1 — Run this to check your work ===

# Recompute expected values
def bayes(prior, tpr, fpr):
    p_pos = tpr * prior + fpr * (1 - prior)
    return (tpr * prior) / p_pos

expected_A = bayes(0.30, 0.95, 0.05)
expected_B = bayes(0.001, 0.95, 0.05)

assert abs(posterior_A - expected_A) < 0.01, \
    f'Scenario A is off. Expected ~{expected_A:.1%}, got {posterior_A:.1%}. Did you change p_disease_A to 0.30?'
assert abs(posterior_B - expected_B) < 0.005, \
    f'Scenario B is off. Expected ~{expected_B:.2%}, got {posterior_B:.2%}. Did you change p_disease_B to 0.001?'

print('✅ Challenge 1 correct!')
print(f'   Scenario A: {posterior_A:.1%} — most positives are real when disease is common')
print(f'   Scenario B: {posterior_B:.2%} — most positives are still false when disease is very rare')

---
## Part 3 — The Prior → Posterior Update Loop

Here's the power move: **the posterior from one update becomes the prior for the next.**

You don't just update once. You update continuously as new data arrives. Each new piece of evidence shifts your belief a bit more.

**Example: Detecting a biased coin**

You're handed a coin. You don't know if it's fair (50/50) or biased toward heads (70/30). You start with a prior belief: 50% chance it's fair, 50% chance it's biased. Now you flip it and watch what happens to your belief.

This is exactly what happens when a stock starts moving differently than expected — you're running a real-time hypothesis test about whether something changed.

In [ ]:
# === CONCEPT: Sequential Bayesian Updating — Coin Bias Detection ===

# Hypotheses:
#   H_fair:   coin is fair, P(heads) = 0.50
#   H_biased: coin is biased, P(heads) = 0.70

p_heads_if_fair   = 0.50
p_heads_if_biased = 0.70

# Start with equal priors — no idea which it is
p_fair_prior   = 0.50
p_biased_prior = 0.50

# Simulate flipping a biased coin (truth: it IS biased)
np.random.seed(42)
n_flips = 30
flips = np.random.binomial(1, p_heads_if_biased, n_flips)  # 1 = heads

# Track posterior belief in the biased hypothesis after each flip
p_biased_history = [p_biased_prior]
p_fair   = p_fair_prior
p_biased = p_biased_prior

for flip in flips:
    # Likelihood of this flip under each hypothesis
    if flip == 1:  # heads
        likelihood_fair   = p_heads_if_fair
        likelihood_biased = p_heads_if_biased
    else:          # tails
        likelihood_fair   = 1 - p_heads_if_fair
        likelihood_biased = 1 - p_heads_if_biased

    # Unnormalized posteriors
    unnorm_fair   = likelihood_fair   * p_fair
    unnorm_biased = likelihood_biased * p_biased

    # Normalize so they sum to 1
    total = unnorm_fair + unnorm_biased
    p_fair   = unnorm_fair   / total
    p_biased = unnorm_biased / total

    p_biased_history.append(p_biased)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(11, 7), gridspec_kw={'height_ratios': [1, 3]})

# Top: show the actual flips
axes[0].scatter(range(1, n_flips+1), flips,
                c=['#e74c3c' if f else '#3498db' for f in flips],
                marker='|', s=200, zorder=3)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Tails', 'Heads'])
axes[0].set_xlim(0, n_flips + 1)
axes[0].set_title('Flip outcomes (coin is actually biased: 70% heads)', fontsize=12)

# Bottom: belief update
axes[1].plot(range(n_flips+1), p_biased_history, color='#e74c3c', linewidth=2.5, label='P(biased)')
axes[1].plot(range(n_flips+1), [1 - p for p in p_biased_history], color='#3498db', linewidth=2.5, label='P(fair)')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='50/50')
axes[1].set_xlabel('Number of flips')
axes[1].set_ylabel('Posterior probability')
axes[1].set_title('Bayesian belief update: is this coin biased?', fontsize=12)
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].annotate(f'After {n_flips} flips: {p_biased_history[-1]:.1%} confident it is biased',
                 xy=(n_flips*0.6, 0.15), fontsize=11,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.tight_layout()
plt.show()

print(f'Final: {p_biased_history[-1]:.1%} confident the coin is biased')

---
### Challenge 2 — Control the Sequence

The simulation above randomly generated flips. Now you're going to manually specify what happens.

**Your task:** Define your own sequence of flips and observe how belief updates.

Try two scenarios:
1. **Fast evidence:** 8 heads in a row, then alternating
2. **Mixed evidence:** 5 heads, 3 tails, 5 heads, 3 tails...

After running each, write a comment answering:
- How quickly does the model become confident?
- What happens to your belief when you get several tails in a row, even if you were already leaning toward "biased"?

In [ ]:
# === CHALLENGE 2 — Modify the flip sequence ===

# Change this list to control the flips (1 = heads, 0 = tails)
# Current: 8 heads then alternating — CHANGE THIS to your own sequences
my_flips = [1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
# Try also: [1,1,1,1,1,1,1,1,0,1,0,0,0,1,0,1,0,1,0,1]

p_fair_c2   = 0.50
p_biased_c2 = 0.50
history_c2  = [p_biased_c2]

for flip in my_flips:
    lk_fair   = p_heads_if_fair   if flip else (1 - p_heads_if_fair)
    lk_biased = p_heads_if_biased if flip else (1 - p_heads_if_biased)
    unnorm_f  = lk_fair   * p_fair_c2
    unnorm_b  = lk_biased * p_biased_c2
    total     = unnorm_f + unnorm_b
    p_fair_c2   = unnorm_f / total
    p_biased_c2 = unnorm_b / total
    history_c2.append(p_biased_c2)

plt.figure(figsize=(10, 4))
plt.plot(history_c2, color='#e74c3c', linewidth=2.5, label='P(biased)')
plt.plot([1-p for p in history_c2], color='#3498db', linewidth=2.5, label='P(fair)')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Flip number')
plt.ylabel('Posterior probability')
plt.title('My custom flip sequence — belief update', fontsize=12)
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

final_c2 = history_c2[-1]
print(f'Final belief in biased: {final_c2:.1%}')

# YOUR OBSERVATIONS:
# How quickly does the model become confident?
# What happens when tails arrive after you were leaning biased?
# [Write your answers here]

In [ ]:
# === ASSERT 2 ===

assert len(my_flips) >= 15, 'Use at least 15 flips so you can observe the pattern'
assert all(f in [0, 1] for f in my_flips), 'Flips must be 0 (tails) or 1 (heads)'
assert 0 < final_c2 < 1, 'Something went wrong — final belief should be between 0 and 1'

n_heads = sum(my_flips)
n_tails = len(my_flips) - n_heads
print(f'✅ Challenge 2 passed!')
print(f'   Your sequence: {n_heads} heads, {n_tails} tails out of {len(my_flips)} flips')
print(f'   Final belief in biased coin: {final_c2:.1%}')

---
## Part 4 — Trading Application: Updating Your Vol Estimate

You're running a volatility book. IBM's 30-day implied vol is trading at **25%** (annualized). That's your **prior** — what the market expects.

Then IBM gaps **3% in a single day**.

Should you revise your vol estimate? By how much?

This is a Bayesian problem. The daily move is your data. You use it to update your estimate of what realized vol actually is.

### Setting up the math

Daily vol is annual vol divided by √252 (trading days):

```
daily_vol = annual_vol / sqrt(252)
```

At 25% annual vol, a "normal" daily move is about **1.6%**. A 3% move is almost **2 standard deviations** — unusual but not impossible.

We'll use a simple Bayesian update rule:
- **Prior:** Gaussian belief centered on the market's implied vol
- **Likelihood:** How probable is today's move under different vol regimes?
- **Posterior:** Updated estimate of true vol given the move we just saw

This is a continuous version of the coin bias problem — instead of two hypotheses, we have a range of possible vol values.

In [ ]:
# === CONCEPT: Bayesian Vol Update ===

# Prior belief about annualized vol
implied_vol   = 0.25    # market-implied vol (your prior center)
prior_std     = 0.05    # uncertainty in your prior (+/- 5% vol)

# Observed daily return
daily_return  = 0.03    # IBM moved 3% today

# Vol grid — range of possible true annual vols we're considering
vol_grid = np.linspace(0.05, 0.80, 500)
daily_vol_grid = vol_grid / np.sqrt(252)

# Prior: Gaussian centered on implied vol
prior = stats.norm.pdf(vol_grid, loc=implied_vol, scale=prior_std)

# Likelihood: how probable is a 3% daily move at each possible vol level?
# Under a normal distribution, the probability of seeing this return is:
likelihood = stats.norm.pdf(daily_return, loc=0, scale=daily_vol_grid)

# Posterior = prior * likelihood (then normalize)
posterior_unnorm = prior * likelihood
posterior = posterior_unnorm / np.trapz(posterior_unnorm, vol_grid)  # normalize

# Find the peak of each distribution
prior_peak     = vol_grid[np.argmax(prior)]
posterior_peak = vol_grid[np.argmax(posterior)]

# Plot
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(vol_grid * 100, prior / np.trapz(prior, vol_grid),
        color='#3498db', linewidth=2.5, linestyle='--', label=f'Prior (centered on implied vol {implied_vol:.0%})')
ax.plot(vol_grid * 100, posterior,
        color='#e74c3c', linewidth=2.5, label=f'Posterior (updated after {daily_return:.0%} move)')

ax.axvline(implied_vol * 100, color='#3498db', linestyle=':', alpha=0.7)
ax.axvline(posterior_peak * 100, color='#e74c3c', linestyle=':', alpha=0.7)

ax.annotate(f'Prior peak: {prior_peak:.0%} vol',
            xy=(prior_peak * 100, prior[np.argmax(prior)] / np.trapz(prior, vol_grid)),
            xytext=(prior_peak * 100 - 12, 18),
            arrowprops=dict(arrowstyle='->', color='#3498db'), color='#3498db', fontsize=11)
ax.annotate(f'Posterior peak: {posterior_peak:.0%} vol',
            xy=(posterior_peak * 100, max(posterior)),
            xytext=(posterior_peak * 100 + 2, max(posterior) * 0.85),
            arrowprops=dict(arrowstyle='->', color='#e74c3c'), color='#e74c3c', fontsize=11)

ax.set_xlabel('Annualized Volatility (%)')
ax.set_ylabel('Probability density')
ax.set_title(f'Bayesian vol update: IBM moves {daily_return:.0%} intraday (prior vol = {implied_vol:.0%})', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 80)
plt.tight_layout()
plt.show()

print(f'Prior estimate:     {prior_peak:.1%} vol')
print(f'Posterior estimate: {posterior_peak:.1%} vol')
print(f'Update:             +{(posterior_peak - prior_peak)*100:.1f} vol points')
print()
print(f'A {daily_return:.0%} single-day move suggests true vol may be higher')
print(f'than what was implied. The posterior shifts right toward higher vol.')

---
### Challenge 3 — Change the Move Size and the Prior

**Your task:** Run three scenarios:

1. **Small move:** IBM only moves 0.5% (below expected daily vol)
2. **Huge move:** IBM gaps 6% (very large — possible earnings surprise)
3. **Different prior:** Set implied vol to 50% (already elevated — maybe pre-earnings). Then observe a 3% move. Does the posterior shift as much?

After each run, write a comment answering:
- Which direction does the posterior shift vs the prior?
- When is the update large vs small?
- In scenario 3, why does a high prior vol make the 3% move less surprising?

In [ ]:
# === CHALLENGE 3 — Modify these parameters and run all three scenarios ===

def bayesian_vol_update(implied_vol, daily_return, prior_std=0.05, label=''):
    """Run a Bayesian vol update and return the posterior peak."""
    vol_grid = np.linspace(0.01, 1.20, 1000)
    daily_vol_grid = vol_grid / np.sqrt(252)
    prior = stats.norm.pdf(vol_grid, loc=implied_vol, scale=prior_std)
    likelihood = stats.norm.pdf(daily_return, loc=0, scale=daily_vol_grid)
    posterior_unnorm = prior * likelihood
    posterior = posterior_unnorm / np.trapz(posterior_unnorm, vol_grid)
    prior_norm = prior / np.trapz(prior, vol_grid)
    posterior_peak = vol_grid[np.argmax(posterior)]
    return vol_grid, prior_norm, posterior, posterior_peak

# --- Scenario 1: Small move ---
ret_1       = 0.03    # <-- CHANGE to 0.005 (0.5% move)
implied_1   = 0.25

# --- Scenario 2: Huge move ---
ret_2       = 0.03    # <-- CHANGE to 0.06 (6% move)
implied_2   = 0.25

# --- Scenario 3: High prior vol + 3% move ---
ret_3       = 0.03
implied_3   = 0.25    # <-- CHANGE to 0.50 (50% implied vol, pre-earnings)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
scenarios = [
    (ret_1, implied_1, 'Scenario 1: Small move'),
    (ret_2, implied_2, 'Scenario 2: Huge move'),
    (ret_3, implied_3, 'Scenario 3: High prior vol'),
]

posterior_peaks = []
for ax, (ret, iv, title) in zip(axes, scenarios):
    vg, pr, po, peak = bayesian_vol_update(iv, ret)
    posterior_peaks.append(peak)
    ax.plot(vg * 100, pr, '--', color='#3498db', linewidth=2, label=f'Prior ({iv:.0%})')
    ax.plot(vg * 100, po, color='#e74c3c', linewidth=2, label=f'Posterior ({peak:.0%})')
    ax.axvline(iv * 100, color='#3498db', linestyle=':', alpha=0.6)
    ax.axvline(peak * 100, color='#e74c3c', linestyle=':', alpha=0.6)
    ax.set_title(f'{title}\nReturn: {ret:.1%}', fontsize=11)
    ax.set_xlabel('Annual Vol (%)')
    ax.set_xlim(0, 120)
    ax.legend(fontsize=9)

plt.suptitle('Bayesian vol update — three scenarios', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

for i, (ret, iv, title) in enumerate(scenarios):
    shift = (posterior_peaks[i] - iv) * 100
    print(f'{title}: prior={iv:.0%}, posterior={posterior_peaks[i]:.0%}, shift={shift:+.1f} vol pts')

# YOUR OBSERVATIONS:
# 1. Which direction does each posterior shift vs prior?
# 2. In which scenario is the update largest?
# 3. Why does scenario 3 show a smaller shift than scenario 2?
# [Write your answers here]

In [ ]:
# === ASSERT 3 ===

assert ret_1 != 0.03 or implied_3 != 0.25, \
    'Looks like you did not change any parameters. Update at least one scenario.'

# Check scenario logic: small move should pull posterior DOWN toward lower vol
if ret_1 == 0.005:
    assert posterior_peaks[0] < 0.25, \
        'A 0.5% move is below expected daily vol — posterior should shift left (lower vol)'
    print('✅ Scenario 1 correct: small move pulls posterior toward lower vol')

# Check scenario 2: big move should push posterior UP
if ret_2 == 0.06:
    assert posterior_peaks[1] > 0.25, \
        'A 6% move is far above expected daily vol — posterior should shift right (higher vol)'
    print('✅ Scenario 2 correct: big move pushes posterior toward higher vol')

# Check scenario 3: high prior vol makes 3% move less surprising
if implied_3 == 0.50:
    shift_3 = abs(posterior_peaks[2] - implied_3)
    assert shift_3 < 0.10, \
        'With 50% prior vol, a 3% move is not very surprising — posterior should stay close to prior'
    print('✅ Scenario 3 correct: high prior vol absorbs the 3% move without large update')

print('\n✅ Challenge 3 complete!')

---
## Reflect — Final Check

Before looking at the recap table, write your own explanation of Bayes' theorem in the cell below. Don't copy anything — write it like you'd explain it to someone on your desk.

Your answer should cover:
1. What is a prior?
2. What is a likelihood?
3. What is a posterior?
4. How does this apply to how you'd think about a big move in an option you're watching?

In [ ]:
# === REFLECT ===
# Write your explanation below as a Python comment or multi-line string.
# Be specific — use the examples from this notebook.

my_explanation = """
Prior:     [YOUR ANSWER]
Likelihood:[YOUR ANSWER]
Posterior: [YOUR ANSWER]
Trading:   [YOUR ANSWER]
"""

print(my_explanation)

In [ ]:
# === FINAL ASSERT — Completion gate ===

placeholder_phrases = ['YOUR ANSWER', '[', ']']
filled_in = not any(p in my_explanation for p in placeholder_phrases)

assert filled_in, \
    'Fill in your explanation above before moving on. This is the most important cell in the notebook.'

assert len(my_explanation.strip()) > 100, \
    'Your explanation is too short. Write at least a few sentences for each section.'

print('✅ Reflection complete. Notebook 02 done.')
print()
print('Update your CONTEXT.md:')
print('  - Mark Notebook 02 status as ✅ Complete')
print('  - Fill in what you covered and what was hard')
print('  - Next session: @CONTEXT.md — ready to work on notebook 03')

---
## Recap Table

| Concept | What it means | Trading relevance |
|---|---|---|
| **Prior** | Your belief before seeing new data | Your vol estimate before today's open |
| **Likelihood** | How probable is the data under your hypothesis? | How surprising is this move given current IV? |
| **Posterior** | Updated belief after combining prior + data | Revised vol estimate after observing the move |
| **Sequential updating** | Posterior becomes prior for next update | You update your view flip by flip — or tick by tick |
| **Strong prior** | Big move is less surprising if prior vol is high | 3% move in an earnings name is normal; in a utility, it's alarming |
| **Base rate** | How common is the event in your prior? | Are you in a high-vol or low-vol regime right now? |

---

**Next: Notebook 03 — Time Series & Volatility**  
You'll measure realized vol, compare it to implied vol, and build intuition for GARCH and vol clustering.